# Demo — prepare spliceosome HepG2 training dataset

Filters the full ENCODE eCLIP 600 nt dataset to a demo-sized training set:

- **Signal**: 9 spliceosome-associated RBPs measured in HepG2 cells
- **Genome**: first `PARAMS_MAX_CHROMS_PER_SPLIT` chromosomes per split (demo; set `None` for full run)
- **Quality**: tiles with no eCLIP signal across the selected RBPs are discarded

**Inputs:**
- `resources/parnet-encore-eclip/600nt_windows.no-one-hot.stripped/encode.filtered.pt`
  (canonical 600 nt source; native-length sequences with `pad_side` in metadata)

**Outputs** (in `results/spliceosome-hepg2/datasets/`):
- `dataset.pt` — filtered stripped dataset (9 tracks, variable-length seqs, `pad_side` in meta)
- `dataset.metadata.yaml` — sidecar: split counts, task names, `total_key`, `seq_len`, …
- `rbp_cts.tsv` — selected RBP names and their track indices in the full 223-RBP dataset
- `tiles.bed` — BED6 with tile coordinates and metadata for all splits

**Next:** open `train_from_pretrained.py.ipynb` to fine-tune a pretrained PARNET model.

## Set-up

### Imports

In [18]:
import pylbsr.notebooks
import pylbsr.misc

import json
import torch
import yaml
import pandas as pd
from functools import partial
from pathlib import Path
from tqdm import tqdm

from parnet_demo_utils import (
    FilteredMultiTaskDataset,
    filter_min_read_count,
    parse_tile_name,
    torch_sparse_to_dense,
)

ModuleNotFoundError: No module named 'parnet_demo_utils'

### Parameters

In [19]:
_notebook_name = "prepare_datasets.py.ipynb"
_notebook_path = f"notebooks/explore/{_notebook_name}"

# ── Output dir ────────────────────────────────────────────────────────────────
PATH_OUTPUT_DIR = "results/spliceosome-hepg2/datasets"

# ── RBP selection ──────────────────────────────────────────────────────────────
# 9 spliceosome-associated RBPs measured in HepG2.
# Selected by intersecting the "Spliceosome" category in resources/metadata/yeo_RBP_annotation.function.csv
# with experiments available in both HepG2 and K562 in the full 223-RBP eCLIP panel.
PARAMS_RBP_SET = {
    "AQR", "BUD13", "EFTUD2", "PRPF8", "RBM22", "SF3B4", "SMNDC1", "U2AF1", "U2AF2",
}
PARAMS_CELL_LINE = "HepG2"

# ── Demo filters ───────────────────────────────────────────────────────────────
PARAMS_MAX_CHROMS_PER_SPLIT = 3   # set to None to use all chromosomes (full run)
PARAMS_MIN_READ_COUNT       = 3   # keep tile if any selected track reaches this eCLIP count

⏱ 0.00 s (00:00:00)


### Initialization

In [20]:
pylbsr.notebooks.enable_cell_timing_metadata(show=True)

logger = pylbsr.misc.init_logger(_notebook_name)

PROJECT_DIR = pylbsr.notebooks.find_project_root_from_notebook_path(_notebook_path)
logger.info(f"Project directory: {PROJECT_DIR}")

# Load filepaths config
with open(PROJECT_DIR / "config/filepaths.yaml") as f:
    filepaths = yaml.safe_load(f)

PATH_SOURCE_PT = filepaths["parnet_encore_eclip"]["data_pt"]
PATH_RBP_TSV   = filepaths["metadata"]["full_rbp_set"]

[16:03:45] INFO - Project directory: /mnt/storage1/workspace/lhofer/parnet--globalclip-head


⏱ 0.01 s (00:00:00)


## Load RBP metadata

Reads the full 223-RBP experiment table bundled with the 600 nt dataset and resolves
each selected RBP to its track index in the full dataset.

In [21]:
_rbp_set_path = PROJECT_DIR / PATH_RBP_TSV
rbps_ct_df = pd.read_csv(_rbp_set_path, sep="\t")
display(rbps_ct_df.head(3))

# Build sorted list of "RBP_CellLine" identifiers for the selected RBPs
target_rbp_cts = sorted([
    f"{rbp}_{PARAMS_CELL_LINE}"
    for rbp in PARAMS_RBP_SET
    if f"{rbp}_{PARAMS_CELL_LINE}" in rbps_ct_df["rbp_ct"].values
])
if len(target_rbp_cts) != len(PARAMS_RBP_SET):
    _missing = PARAMS_RBP_SET - {rbp_ct.rsplit("_", 1)[0] for rbp_ct in target_rbp_cts}
    logger.warning(f"RBPs not found in {PARAMS_CELL_LINE}: {_missing}")

# Track index = row position in the 223-RBP table (used to slice signal tensors)
track_indices = [rbps_ct_df["rbp_ct"].tolist().index(rbp_ct) for rbp_ct in target_rbp_cts]

print(f"\nSelected {len(target_rbp_cts)} RBPs in {PARAMS_CELL_LINE}:")
for rbp_ct, idx in zip(target_rbp_cts, track_indices):
    print(f"  track {idx:3d}  {rbp_ct}")

print(target_rbp_cts)

,rbp_ct,rbp,ct
0,AARS_K562,AARS,K562
1,AATF_K562,AATF,K562
2,ABCF1_K562,ABCF1,K562



Selected 9 RBPs in HepG2:
  track   9  AQR_HepG2
  track  13  BUD13_HepG2
  track  41  EFTUD2_HepG2
  track 131  PRPF8_HepG2
  track 144  RBM22_HepG2
  track 159  SF3B4_HepG2
  track 165  SMNDC1_HepG2
  track 193  U2AF1_HepG2
  track 195  U2AF2_HepG2
['AQR_HepG2', 'BUD13_HepG2', 'EFTUD2_HepG2', 'PRPF8_HepG2', 'RBM22_HepG2', 'SF3B4_HepG2', 'SMNDC1_HepG2', 'U2AF1_HepG2', 'U2AF2_HepG2']
⏱ 0.02 s (00:00:00)


## Load train / val / test split config

In [22]:
_splits_cfg_path = PROJECT_DIR / "config" / "config.train_validation_test_split.yaml"
_data_splits = yaml.safe_load(_splits_cfg_path.read_text())

# Config uses "validation"; .pt splits use "valid"
_split_key_map = {"train": "train", "validation": "valid", "test": "test"}

# Demo: keep only the first N chromosomes per split (sorted alphabetically).
# Set PARAMS_MAX_CHROMS_PER_SPLIT = None to use all chromosomes.
filter_chromosomes: dict[str, list[str]] = {
    _split_key_map[cfg_key]: sorted(chroms)[:PARAMS_MAX_CHROMS_PER_SPLIT]
    for cfg_key, chroms in _data_splits.items()
}
for split, chroms in filter_chromosomes.items():
    print(f"  {split:6s}: {chroms}")

  test  : ['chr3', 'chr8']
  valid : ['chr16', 'chr2', 'chr9']
  train : ['chr1', 'chr10', 'chr11']
⏱ 0.01 s (00:00:00)


## Filter and assemble dataset

Loads the full 600 nt stripped `.pt` (memory-mapped) and applies two filters per split:

1. **Chromosome filter** — retain only tiles on the demo-subset chromosomes
2. **Quality filter** — discard tiles with fewer than `PARAMS_MIN_READ_COUNT` eCLIP reads
   across all selected tracks

In [23]:
logger.info(f"Loading (mmap) {PATH_SOURCE_PT} ...")
data = torch.load(Path(PATH_SOURCE_PT), mmap=True, weights_only=False)
logger.info(f"Source splits: { {k: len(v) for k, v in data.items()} }")

[16:03:49] INFO - Loading (mmap) /mnt/storage1/ml4rg26-shared/parnet-eclip/data-formatted-for-training/600nt_windows.no-one-hot.stripped/encode.filtered.pt ...
[16:13:38] INFO - Source splits: {'train': 512946, 'test': 70626, 'valid': 116542}


⏱ 589.06 s (00:09:49)


In [24]:
_quality_filter = partial(
    filter_min_read_count,
    min_read_count=PARAMS_MIN_READ_COUNT,
    tasks=["eCLIP"],
    track_indices=track_indices,
)

output_splits: dict[str, list] = {}

for split_name, target_chroms in filter_chromosomes.items():
    if split_name not in data:
        logger.warning(f"Split '{split_name}' not found in source; skipping.")
        continue
    _chrom_set = set(target_chroms)

    # Step 1: chromosome filter (fast list comprehension before FilteredMultiTaskDataset)
    _in_chroms = [
        s for s in data[split_name]
        if parse_tile_name(s["meta"]["name"])[0] in _chrom_set
    ]
    logger.info(
        f"[{split_name}] {len(data[split_name]):,} → {len(_in_chroms):,} tiles on {target_chroms}"
    )

    # Step 2: quality filter + track slicing via FilteredMultiTaskDataset
    _ds = FilteredMultiTaskDataset(_in_chroms, track_indices, [_quality_filter])
    output_splits[split_name] = [_ds[i] for i in tqdm(range(len(_ds)), desc=split_name)]
    logger.info(f"[{split_name}] {len(output_splits[split_name]):,} tiles after quality filter")

logger.info("Done.")

[16:33:38] INFO - [test] 70,626 → 70,626 tiles on ['chr3', 'chr8']
test: 100%|██████████| 22388/22388 [00:11<00:00, 2008.88it/s]
[16:34:01] INFO - [test] 22,388 tiles after quality filter
[16:34:01] INFO - [valid] 116,542 → 116,542 tiles on ['chr16', 'chr2', 'chr9']
valid: 100%|██████████| 45849/45849 [00:23<00:00, 1920.49it/s]
[16:34:48] INFO - [valid] 45,849 tiles after quality filter
[16:34:48] INFO - [train] 512,946 → 125,802 tiles on ['chr1', 'chr10', 'chr11']
train: 100%|██████████| 42367/42367 [05:30<00:00, 128.11it/s]
[16:45:42] INFO - [train] 42,367 tiles after quality filter
[16:45:42] INFO - Done.


⏱ 725.01 s (00:12:05)


## Save output dataset

In [25]:
output_dir = PROJECT_DIR / PATH_OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)
output_pt_path = output_dir / "dataset.pt"

logger.info(f"Saving → {output_pt_path} ...")
torch.save(output_splits, output_pt_path)
logger.info(f"Saved ({output_pt_path.stat().st_size / 1e6:.0f} MB)")

# Metadata sidecar — always set total_key="eCLIP" explicitly (not inferred from dict order)
_task_names = list(output_splits[list(output_splits.keys())[0]][0]["outputs"].keys())
_meta = {
    "sequence_format": "string",
    "signal_format":   "sparse_tensor",
    "seq_len":         600,
    "n_tracks":        len(target_rbp_cts),
    "task_names":      _task_names,
    "total_key":       "eCLIP",
    "splits":          {split: len(samples) for split, samples in output_splits.items()},
    "source":          PATH_SOURCE_PT,
    "is_pre_padded":   False,
}
_meta_path = output_dir / "dataset.metadata.yaml"
_meta_path.write_text(yaml.dump(_meta, default_flow_style=False))
logger.info(f"Metadata → {_meta_path}")

[16:46:44] INFO - Saving → /mnt/storage1/workspace/lhofer/parnet--globalclip-head/results/spliceosome-hepg2/datasets/dataset.pt ...
[16:47:06] INFO - Saved (823 MB)
[16:47:06] INFO - Metadata → /mnt/storage1/workspace/lhofer/parnet--globalclip-head/results/spliceosome-hepg2/datasets/dataset.metadata.yaml


⏱ 22.04 s (00:00:22)


In [26]:
# Export rbp_cts.tsv — RBP table with track indices in the full 223-RBP dataset.
# Used by train_from_pretrained and evaluate_retrained_models notebooks.
_rbp_cts_path = output_dir / "rbp_cts.tsv"
_rbp_out_df = rbps_ct_df[rbps_ct_df["rbp_ct"].isin(target_rbp_cts)].copy().reset_index(drop=True)
_rbp_out_df["track_index_in_full_dataset"] = [
    track_indices[target_rbp_cts.index(rbp_ct)]
    for rbp_ct in _rbp_out_df["rbp_ct"]
]
_rbp_out_df.to_csv(_rbp_cts_path, sep="\t", index=False)
logger.info(f"RBP table → {_rbp_cts_path}")
display(_rbp_out_df)

[16:47:10] INFO - RBP table → /mnt/storage1/workspace/lhofer/parnet--globalclip-head/results/spliceosome-hepg2/datasets/rbp_cts.tsv


,rbp_ct,rbp,ct,track_index_in_full_dataset
0,AQR_HepG2,AQR,HepG2,9
1,BUD13_HepG2,BUD13,HepG2,13
2,EFTUD2_HepG2,EFTUD2,HepG2,41
3,PRPF8_HepG2,PRPF8,HepG2,131
4,RBM22_HepG2,RBM22,HepG2,144
5,SF3B4_HepG2,SF3B4,HepG2,159
6,SMNDC1_HepG2,SMNDC1,HepG2,165
7,U2AF1_HepG2,U2AF1,HepG2,193
8,U2AF2_HepG2,U2AF2,HepG2,195


⏱ 0.09 s (00:00:00)


In [27]:
# Export tiles.bed — BED6 with tile coordinates, pad_side score, and JSON metadata in name column.
# Score column = pad_side: -1 (full window), 0 (center), 1 (left), 2 (right).
_bed_path = output_dir / "tiles.bed"
_n_bed = 0
with open(_bed_path, "w") as _f:
    for split_name, samples in output_splits.items():
        for elem in samples:
            name = elem["meta"]["name"]

            if isinstance(name, bytes):
                name = name.decode("utf-8")

            chrom, start, end, strand = parse_tile_name(name)

            score = elem["meta"].get("pad_side", -1)

            extra_meta = {k: v for k, v in elem["meta"].items() if k != "name"}
            extra_meta["split"] = split_name

            name_col = f"{name};{json.dumps(extra_meta, sort_keys=True, separators=(',', ':'))}"

            _f.write(f"{chrom}\t{start}\t{end}\t{name_col}\t{score}\t{strand}\n")
            _n_bed += 1

logger.info(f"BED → {_bed_path}  ({_n_bed:,} rows)")

[16:47:16] INFO - BED → /mnt/storage1/workspace/lhofer/parnet--globalclip-head/results/spliceosome-hepg2/datasets/tiles.bed  (110,604 rows)


⏱ 0.66 s (00:00:00)


## Sanity check

Quick reload to verify shapes and metadata.

In [28]:
_check = torch.load(output_pt_path, mmap=True, weights_only=False)
_s = _check["train"][0]

print(f"sequence length  : {len(_s['inputs']['sequence'])} nt  (variable; <= 600)")
print(f"pad_side         : {_s['meta'].get('pad_side')}")

for task, sparse in _s["outputs"].items():
    _dense = torch_sparse_to_dense(sparse)
    print(f"output '{task}'  shape : {tuple(_dense.shape)}  (expected ({len(target_rbp_cts)}, ...))")

print()
print("Split counts:")

for split, samples in _check.items():
    print(f"  {split:6s}: {len(samples):,}")

del _check

sequence length  : 600 nt  (variable; <= 600)
pad_side         : -1
output 'control'  shape : (9, 600)  (expected (9, ...))
output 'eCLIP'  shape : (9, 600)  (expected (9, ...))

Split counts:
  test  : 22,388
  valid : 45,849
  train : 42,367
⏱ 39.09 s (00:00:39)


## Converting the output to HFDS format

If downstream tools prefer HuggingFace Arrow format, convert the filtered `.pt` with
the dedicated script (no notebook required):

```bash
pixi run -e parnet-dev-cu12 python scripts/convert_pt_to_hfds.py \
    --input     results/spliceosome-hepg2/datasets/dataset.pt \
    --outputdir results/spliceosome-hepg2/datasets/dataset.hfds \
    --total-key eCLIP \
    --is-pre-padded false \
    --seq-len 600
```

The resulting `dataset.hfds/` is accepted by the `HFDSDataset` loader in
`train_from_pretrained.py.ipynb` (set `params_dataset_format = "hfds"`).